In [1]:
import pandas as pd
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

In [2]:
X_train = pd.read_csv("../data/X_train.csv")
X_test = pd.read_csv("../data/X_test.csv")

y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_test = pd.read_csv("../data/y_test.csv").squeeze()

In [3]:
print(X_train.shape)
print(X_test.shape)

(367, 27)
(92, 27)


In [4]:
from collections import Counter

class_counts = Counter(y_train)
scale_pos_weight = class_counts[0] / class_counts[1]

print("Class counts:", class_counts)
print("XGBoost scale_pos_weight:", scale_pos_weight)

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42,
        class_weight="balanced"
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        min_samples_leaf=2
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight
    )
}

Class counts: Counter({0: 275, 1: 92})
XGBoost scale_pos_weight: 2.989130434782609


In [5]:
results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    prediction = model.predict(X_test)
    probability = model.predict_proba(X_test)[:,1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, prediction),
        "Precision": precision_score(y_test, prediction),
        "Recall": recall_score(y_test, prediction),
        "F1": f1_score(y_test, prediction),
        "ROC_AUC": roc_auc_score(y_test, probability)
    })

results_df = pd.DataFrame(results)
results_df

/Applications/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.913043,0.800000,0.869565,0.833333,0.969124
1,Random Forest,0.967391,1.000000,0.869565,0.930233,0.989918
2,XGBoost,0.945652,0.909091,0.869565,0.888889,0.982357


In [6]:
results_df = results_df.sort_values(
    by="ROC_AUC",
    ascending=False
)

results_df

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
1,Random Forest,0.967391,1.000000,0.869565,0.930233,0.989918
2,XGBoost,0.945652,0.909091,0.869565,0.888889,0.982357
0,Logistic Regression,0.913043,0.800000,0.869565,0.833333,0.969124


In [7]:
best_model_name = results_df.iloc[0]["Model"]

print(best_model_name)

Random Forest


In [8]:
best_model = models[best_model_name]

joblib.dump(
    best_model,
    "../models/best_model.pkl"
)

print("Best model saved!")

Best model saved!


In [9]:
results_df.to_csv(
    "../data/model_results.csv",
    index=False
)